# Notebook 2 — SLM Fine-Tuning (QLoRA)

**Model:** `meta-llama/Llama-3.2-3B-Instruct`  
**Method:** QLoRA (4-bit NF4 quantization + LoRA adapters)  
**Framework:** HuggingFace Transformers + PEFT + TRL  

---

## Model Selection Rationale

| Criterion | Decision |
|-----------|----------|
| Parameter limit | Llama-3.2-3B is **exactly at 3B** — uses the full budget |
| Base quality | Already instruction-tuned → LoRA teaches domain, not instruction-following |
| Before-vs-after delta | 3B base without domain tuning will visibly fail on DPD/FOIR — measurable improvement |
| Memory footprint | ~2.5 GB at 4-bit — trains on 8 GB VRAM |
| Ecosystem | Best PEFT/TRL integration; chat template natively supported |

**Fallback model** (if HuggingFace gating is an issue): `microsoft/phi-2` (2.7B, ungated, strong reasoning)

## Why QLoRA (not full fine-tuning)?

Full fine-tuning of a 3B model requires ~24 GB VRAM and hours of training time. QLoRA:
- Freezes the base model weights (quantized to 4-bit NF4)
- Trains only small LoRA adapter matrices (~0.1% of total parameters)
- Reduces VRAM requirement to ~6–8 GB
- Completes training on 2,400 examples in 15–25 minutes on a 12 GB GPU
- Preserves the base model's language understanding while injecting domain knowledge


In [ ]:
# Environment check
import torch
import transformers
import peft
import trl
import bitsandbytes as bnb

print(f'PyTorch:        {torch.__version__}')
print(f'Transformers:   {transformers.__version__}')
print(f'PEFT:           {peft.__version__}')
print(f'TRL:            {trl.__version__}')
print(f'BitsAndBytes:   {bnb.__version__}')
print()

if torch.cuda.is_available():
    device = torch.cuda.get_device_name(0)
    vram   = torch.cuda.get_device_properties(0).total_memory / 1e9
    bf16_support = torch.cuda.is_bf16_supported()
    print(f'GPU:    {device}')
    print(f'VRAM:   {vram:.1f} GB')
    print(f'BF16:   {"Supported" if bf16_support else "Not supported (will use FP16)"}')
else:
    print('WARNING: No CUDA GPU detected. Training will be very slow on CPU.')

## Step 1: Load Quantized Base Model

In [ ]:
import json
import yaml
from pathlib import Path
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

# Load config
with open('../configs/training_config.yaml') as f:
    config = yaml.safe_load(f)

MODEL_NAME = config['model']['name']
print(f'Model: {MODEL_NAME}')

In [ ]:
# 4-bit NF4 quantization configuration
# NF4 (Normal Float 4) is theoretically optimal for normally distributed weights
# Double quantization saves ~0.4 bits per parameter with no accuracy loss

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

print(f'Quantization: 4-bit NF4, double quant, compute_dtype={compute_dtype}')

# Load model — first run will download ~1.5 GB
print(f'\nLoading {MODEL_NAME}...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=compute_dtype,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print(f'Model loaded. Vocab size: {tokenizer.vocab_size:,}')

## Step 2: Apply LoRA Adapters

**LoRA hyperparameter choices:**

- `r=16`: Rank of the adapter matrices. Higher rank = more capacity but more parameters.
  With 2,400 training examples, r=16 is sufficient — r=64 risks overfitting.
- `lora_alpha=32`: Scaling factor. alpha/r = 2.0 is the standard initialization that
  prevents the adapter from dominating the base model at initialization.
- `lora_dropout=0.05`: Small dropout for regularization on our small dataset.
- `target_modules`: We adapt all 4 attention projection layers. This gives the model
  the most opportunity to learn new attention patterns over lending concepts, while
  keeping total trainable parameters small.

In [ ]:
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.config.use_cache = False  # required for gradient checkpointing

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)

model = get_peft_model(model, lora_config)

trainable, total = model.get_nb_trainable_parameters()
print(f'Trainable parameters: {trainable:,}')
print(f'Total parameters:     {total:,}')
print(f'Trainable fraction:   {100 * trainable / total:.4f}%')
print()
print('Base model weights are FROZEN. Only LoRA adapters will be updated.')

## Step 3: Load Training Data

In [ ]:
def load_jsonl(path):
    records = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return Dataset.from_list(records)

train_dataset = load_jsonl('../data/processed/train.jsonl')
val_dataset   = load_jsonl('../data/processed/val.jsonl')

print(f'Training examples:   {len(train_dataset)}')
print(f'Validation examples: {len(val_dataset)}')

# Preview first example structure
print(f'\nExample structure (keys): {list(train_dataset[0].keys())}')
print(f'Message roles: {[m["role"] for m in train_dataset[0]["messages"]]}')

## Step 4: Configure and Run Training

In [ ]:
import os
os.makedirs('../outputs/adapter', exist_ok=True)

use_bf16 = torch.cuda.is_bf16_supported()
use_fp16 = not use_bf16

# Training arguments — see configs/training_config.yaml for rationale
training_args = TrainingArguments(
    output_dir='../outputs/adapter',
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,     # effective batch = 16
    gradient_checkpointing=True,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_steps=100,
    weight_decay=0.001,
    fp16=use_fp16,
    bf16=use_bf16,
    evaluation_strategy='steps',
    eval_steps=100,
    save_strategy='steps',
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    logging_steps=25,
    report_to='none',
    seed=42,
)

# Formatting function: convert messages list to chat-templated string
def formatting_func(example):
    return [
        tokenizer.apply_chat_template(
            ex['messages'],
            tokenize=False,
            add_generation_prompt=False,
        )
        for ex in example
    ]

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    formatting_func=formatting_func,
    max_seq_length=512,
    args=training_args,
    peft_config=lora_config,
)

print('Trainer configured. Ready to train.')
print(f'Steps per epoch:    ~{len(train_dataset) // (4 * 4)}')
print(f'Total steps (2 ep): ~{(len(train_dataset) // (4 * 4)) * 2}')

In [ ]:
import time

print('Starting QLoRA fine-tuning...')
print('Expected time: 15–25 minutes on 12 GB GPU')
print()

start = time.time()
train_result = trainer.train()
elapsed = time.time() - start

print(f'\nTraining complete in {elapsed/60:.1f} minutes')
print(f'Final train loss: {train_result.metrics.get("train_loss", "N/A"):.4f}')

## Step 5: Save Adapter Checkpoint

In [ ]:
import json

trainer.model.save_pretrained('../outputs/adapter')
tokenizer.save_pretrained('../outputs/adapter')

metrics = train_result.metrics
metrics['elapsed_minutes'] = elapsed / 60
metrics['model_name'] = MODEL_NAME

import os
os.makedirs('../outputs', exist_ok=True)
with open('../outputs/training_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2, default=str)

print('Adapter saved to ../outputs/adapter/')
print('Training metrics saved to ../outputs/training_metrics.json')

# List saved files
import os
saved = os.listdir('../outputs/adapter')
print(f'\nSaved files: {saved}')

## Step 6: Inspect Training Metrics

In [ ]:
# Plot training and validation loss
try:
    import matplotlib.pyplot as plt

    log_history = trainer.state.log_history

    train_steps  = [x['step'] for x in log_history if 'loss' in x and 'eval_loss' not in x]
    train_losses = [x['loss'] for x in log_history if 'loss' in x and 'eval_loss' not in x]
    eval_steps   = [x['step'] for x in log_history if 'eval_loss' in x]
    eval_losses  = [x['eval_loss'] for x in log_history if 'eval_loss' in x]

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(train_steps, train_losses, label='Train Loss', alpha=0.8, color='#3B6EFF')
    ax.plot(eval_steps,  eval_losses,  label='Val Loss',   alpha=0.9, color='#F59E0B', linewidth=2, marker='o')
    ax.set_xlabel('Training Steps')
    ax.set_ylabel('Loss')
    ax.set_title('QLoRA Fine-Tuning — Training & Validation Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('../outputs/training_loss.png', dpi=150)
    plt.show()
    print('Loss curve saved to ../outputs/training_loss.png')
except Exception as e:
    print(f'Plot skipped: {e}')

In [ ]:
# Quick sanity check — run one inference with the fine-tuned model
from peft import PeftModel

test_prompt = [
    {"role": "system", "content": "You are a Lending Intelligence Assistant at ABC Finance Ltd."},
    {"role": "user",   "content": (
        "Classify the credit risk for this borrower:\n\n"
        "Bureau Score: 624 | Current DPD: 30 | Max DPD: 90\n"
        "FOIR: 0.68 | Monthly Income: ₹42,000 | Collection Bucket: 1-30\n"
        "Default Flag: No | Loan Product: Personal Loan"
    )},
]

prompt_str = tokenizer.apply_chat_template(
    test_prompt, tokenize=False, add_generation_prompt=True
)
inputs = tokenizer(prompt_str, return_tensors='pt').to(model.device)

import torch
with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )

new_tokens = output[0][inputs['input_ids'].shape[1]:]
response = tokenizer.decode(new_tokens, skip_special_tokens=True)

print('Fine-tuned model response:')
print('-' * 50)
print(response)
print('-' * 50)
print('\nExpected: HIGH RISK with reasoning citing Bureau<650, DPD>0, FOIR>0.60')

---

## Summary

| Component | Detail |
|-----------|--------|
| Base model | Llama-3.2-3B-Instruct |
| Quantization | 4-bit NF4 + double quantization |
| LoRA rank | r=16, alpha=32, dropout=0.05 |
| Target modules | q_proj, k_proj, v_proj, o_proj |
| Trainable params | ~2.6M (0.08% of 3B) |
| Training data | 2,400 examples (800 records × 3 tasks) |
| Epochs | 2 |
| Effective batch | 16 (4 × 4 grad accum) |
| Learning rate | 2e-4 cosine with 100 warmup steps |
| Peak VRAM | ~6.5 GB |

**Next:** `03_evaluation.ipynb` — before-vs-after comparison and business impact assessment
